# Approach C: fine-tune WangchanBERTa

Fine-tunes the pretrained Thai model [WangchanBERTa](https://huggingface.co/airesearch/wangchanberta-base-att-spm-uncased)
(`airesearch/wangchanberta-base-att-spm-uncased`) on `train.csv`. It uses the same splits and the same `evaluate()` /
`save_submission()` from `code/common.py` as Approaches A and B, so all results are comparable.

**How to run in Google Colab**
1. Runtime → Change runtime type → **T4 GPU**
2. Runtime → Run all. Each experiment takes roughly 10–20 minutes on a T4 (3 epochs).
3. The last cell downloads a zip with `results/` and `predictions/`. Colab deletes files when the runtime ends, so download the zip before you close it.

Ablation (one change per step):

| Step | Change |
|---|---|
| C1 | WangchanBERTa fine-tuned with plain cross-entropy |
| C2 | C1 + class weights `balanced` in the loss (same idea as A2) |

The best epoch of each run is chosen on **valid** macro-F1. Test text is only passed through our own fine-tuned model, never to any external service.

## 1. Get the repository

In [ ]:
import os, sys

REPO_URL = "https://github.com/Passakorn-Aiamwasu/EGCO461-Project-Thai-Sentiment-Classification.git"
BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("/content/repo"):
        # If the repo is private this fails: upload a zip of the repo instead and unzip it to /content/repo
        !git clone -q --branch {BRANCH} {REPO_URL} /content/repo
    %cd /content/repo
elif os.path.basename(os.getcwd()) == "code":
    os.chdir("..")  # running locally from code/: move to the repo root

assert os.path.exists("data/train.csv") and os.path.exists("code/common.py"), "run from the repo root"

In [ ]:
# Colab already has torch and transformers
!pip install -q pythainlp sentencepiece transformers

## 2. Imports and settings

In [ ]:
import copy, html, random, re, time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          get_linear_schedule_with_warmup)

sys.path.insert(0, "code")
from common import GROUP, LABELS, PRED_DIR, RESULTS_DIR, evaluate, load_data, save_submission

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"  # mixed precision: faster on GPU
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
if DEVICE == "cpu":
    print("WARNING: no GPU. Switch the Colab runtime to T4 GPU, training on CPU takes many hours.")

In [ ]:
MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"
MAX_LEN = 128        # ~97% of train texts fit; longer texts are truncated
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
LR = 3e-5
EPOCHS = 3
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
SEED = 42

# name -> settings. Each experiment is one full fine-tuning run.
EXPERIMENTS = {
    "C1_wangchanberta": dict(class_weight=None),
    "C2_wangchanberta_balanced": dict(class_weight="balanced"),
}

# Optional: after choosing the best experiment on valid, retrain it on train+valid
# for its best number of epochs (allowed by the rules). Doubles the run time.
RETRAIN_ON_TRAIN_VALID = False

LABEL2ID = {l: i for i, l in enumerate(LABELS)}


def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

## 3. Preprocessing

WangchanBERTa was pretrained on text where spaces are written as the token `<_>` and text is lowercased.
We apply the same idea and also collapse characters repeated 3+ times (มากกกก → มาก).

In [ ]:
REPEAT_RE = re.compile(r"(.)\1{2,}")


def preprocess(text):
    text = html.unescape(str(text)).lower()
    text = REPEAT_RE.sub(r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.replace(" ", "<_>")


def load_tokenizer():
    try:
        return AutoTokenizer.from_pretrained(MODEL_NAME)
    except Exception as e:  # fallback: the original sentencepiece tokenizer class of WangchanBERTa
        print("AutoTokenizer failed, using CamembertTokenizer:", e)
        from transformers import CamembertTokenizer
        return CamembertTokenizer.from_pretrained(MODEL_NAME)


def load_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=LABEL2ID)


train, valid, sample, test = load_data()
print(preprocess("อร่อยมากกกก  ร้านนี้ 555"))

## 4. Tokenize

In [ ]:
tokenizer = load_tokenizer()
collator = DataCollatorWithPadding(tokenizer)


def encode(df, with_labels=True):
    enc = tokenizer([preprocess(t) for t in df["text"]], truncation=True, max_length=MAX_LEN)
    items = [{"input_ids": ids, "attention_mask": am} for ids, am in zip(enc["input_ids"], enc["attention_mask"])]
    if with_labels:
        for item, label in zip(items, df["category"]):
            item["labels"] = LABEL2ID[label]
    return items


data = {"train": encode(train), "valid": encode(valid), "sample": encode(sample), "test": encode(test, False)}
lengths = [len(x["input_ids"]) for x in data["train"]]
print({k: len(v) for k, v in data.items()},
      f"| tokens per text: median {int(np.median(lengths))}, truncated {np.mean(np.array(lengths) >= MAX_LEN):.1%}")

## 5. Training loop

In [ ]:
def class_weights(labels, mode):
    if mode is None:
        return None
    counts = np.bincount([LABEL2ID[l] for l in labels], minlength=len(LABELS))
    w = len(labels) / (len(LABELS) * counts)  # same formula as sklearn class_weight='balanced'
    return torch.tensor(w, dtype=torch.float, device=DEVICE)


@torch.no_grad()
def predict_proba(model, items):
    model.eval()
    loader = DataLoader([{k: v for k, v in x.items() if k != "labels"} for x in items],
                        batch_size=EVAL_BATCH_SIZE, collate_fn=collator)
    out = []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(**batch).logits
        out.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    return np.concatenate(out)


def to_labels(proba):
    return np.array(LABELS)[proba.argmax(axis=1)]


def fine_tune(cfg, train_items, train_labels, epochs=EPOCHS, valid_df=None, valid_items=None):
    # Train one model. With valid data: keep the epoch with the best valid macro-F1.
    set_seed()
    model = load_model().to(DEVICE)
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights(train_labels, cfg["class_weight"]))
    loader = DataLoader(train_items, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator,
                        generator=torch.Generator().manual_seed(SEED))
    no_decay = ["bias", "LayerNorm.weight"]
    params = [{"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
               "weight_decay": WEIGHT_DECAY},
              {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
               "weight_decay": 0.0}]
    optimizer = torch.optim.AdamW(params, lr=LR)
    total_steps = len(loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    history, best_f1, best_state, best_epoch = [], -1.0, None, epochs
    for epoch in range(1, epochs + 1):
        model.train()
        t0, running = time.time(), 0.0
        for step, batch in enumerate(loader, 1):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = batch.pop("labels")
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = model(**batch).logits
            loss = loss_fn(logits.float(), labels)
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running += loss.item()
            if step % 200 == 0:
                print(f"  epoch {epoch} step {step}/{len(loader)} loss {running / step:.4f}")
        row = {"epoch": epoch, "train_loss": running / len(loader), "minutes": (time.time() - t0) / 60}
        if valid_items is not None:
            f1 = evaluate("", {"valid": (valid_df["category"], to_labels(predict_proba(model, valid_items)))})["valid_macroF1"]
            row["valid_macroF1"] = f1
            if f1 > best_f1:
                best_f1, best_epoch = f1, epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        history.append(row)
        print(row)
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_epoch, history

## 6. Run the experiments

In [ ]:
RESULTS_DIR.mkdir(exist_ok=True)
PRED_DIR.mkdir(exist_ok=True)

rows, histories, best_epochs, test_proba = [], [], {}, {}
valid_out = valid[["id", "text", "category"]].copy()
for name, cfg in EXPERIMENTS.items():
    print(f"=== {name} {cfg}")
    model, best_epoch, history = fine_tune(cfg, data["train"], train["category"],
                                           valid_df=valid, valid_items=data["valid"])
    best_epochs[name] = best_epoch
    histories += [{"experiment": name, **h} for h in history]

    proba = {split: predict_proba(model, data[split]) for split in ["valid", "sample", "test"]}
    row = evaluate(name, {"valid": (valid["category"], to_labels(proba["valid"])),
                          "sample": (sample["category"], to_labels(proba["sample"]))})
    row["best_epoch"] = best_epoch
    rows.append(row)
    print(f"{name}: valid={row['valid_macroF1']:.3f} sample={row['sample_macroF1']:.3f} (best epoch {best_epoch})")

    # Save probabilities for ensembles later. Test probabilities go to predictions/ (not committed).
    valid_out[name] = to_labels(proba["valid"])
    for split, df in [("valid", valid), ("sample", sample)]:
        pd.DataFrame(proba[split], columns=LABELS).assign(id=df["id"].values).to_csv(
            RESULTS_DIR / f"approach_c_{name}_{split}_proba.csv", index=False)
    test_proba[name] = proba["test"]
    pd.DataFrame(proba["test"], columns=LABELS).assign(id=test["id"].values).to_csv(
        PRED_DIR / f"approach_c_{name}_test_proba.csv", index=False)

    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

results = pd.DataFrame(rows).set_index("experiment")
results.round(4).to_csv(RESULTS_DIR / "approach_c_results.csv")
pd.DataFrame(histories).round(4).to_csv(RESULTS_DIR / "approach_c_history.csv", index=False)
valid_out.to_csv(RESULTS_DIR / "approach_c_valid_predictions.csv", index=False)
results.round(3)

## 7. Write the test prediction file (best experiment on valid)

In [ ]:
best = results["valid_macroF1"].idxmax()
print("selected by valid macro-F1:", best)

if RETRAIN_ON_TRAIN_VALID:
    full = pd.concat([train, valid], ignore_index=True)
    model, _, _ = fine_tune(EXPERIMENTS[best], data["train"] + data["valid"], full["category"],
                            epochs=best_epochs[best])
    final_proba = predict_proba(model, data["test"])
    filename = f"{GROUP}_run3_wangchanberta_full.csv"
else:
    final_proba = test_proba[best]
    filename = f"{GROUP}_run3_wangchanberta.csv"

test_pred = to_labels(final_proba)
path = save_submission(test["id"], test_pred, filename)
print("saved", path)
pd.Series(test_pred).value_counts(normalize=True).round(3)

## 8. Download the results (Colab)

In [ ]:
!zip -qr approach_c_outputs.zip results predictions
from google.colab import files
files.download("approach_c_outputs.zip")